# 工程化分步复现手写数字识别

这个 notebook 是工程主入口，用来分步调用 `src/` 模块完成训练、评估和预测。它不作为提交给老师的独立材料；提交材料仍然保留 `submission_notebook.ipynb` 的自包含版本。

默认输出到 `outputs_runs/project_notebook`，不会覆盖上一版 99.8% 高分结果目录 `outputs_submission/`。

In [ ]:
from pathlib import Path
import sys

import torch

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path(r"E:\ALL\学习\AI导论作业-识别手写数字")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import ExperimentConfig, ensure_project_paths
from src.data import create_dataloaders
from src.engine import fit
from src.evaluate import collect_predictions, load_model_from_checkpoint, save_evaluation_bundle
from src.model import build_model, count_model_parameters
from src.predict import PredictionImageDataset, predict_with_tta, write_predictions_csv
from src.train import set_seed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PROJECT_ROOT, DEVICE

## 运行开关

按需打开不同阶段。常见用法：

- 只训练：`RUN_TRAINING=True`
- 只评估旧模型：`LOAD_EXISTING_BEST_MODEL=True`, `RUN_EVALUATION=True`
- 只预测考试图片：设置 `EXAM_IMAGE_DIR`，并打开 `RUN_PREDICTION=True`
- 调参可单独放在后续 HPO 入口，不作为本 notebook 必跑步骤。

In [ ]:
RUN_TRAINING = False
RUN_EVALUATION = True
RUN_PREDICTION = False
LOAD_EXISTING_BEST_MODEL = True

HIGH_SCORE_CHECKPOINT = PROJECT_ROOT / "outputs_submission" / "checkpoints" / "best_model_state.pt"
EXAM_IMAGE_DIR = PROJECT_ROOT / "exam_data" / "test"

config = ExperimentConfig(
    project_root=PROJECT_ROOT,
    run_name="project_notebook",
    dataset_name="mnist",
    model_name="medium_cnn",
    batch_size=64,
    epochs=1,
    dropout=0.21672530847241062,
    optimizer_type="AdamW",
    scheduler_type="CosineAnnealingLR",
    learning_rate=0.0008398721379146775,
    weight_decay=6.602542933207749e-06,
    label_smoothing=0.03,
    max_samples=512,
    use_tta=True,
    tta_n=8,
)
paths = ensure_project_paths(config)
set_seed(config.seed)
config.to_dict()

## 分步训练、评估、预测

下面的 cell 可以分开运行。默认使用小样本 `max_samples=512` 做 smoke test；完整训练时再关闭子采样并调整数据源。

In [ ]:
train_loader, val_loader = create_dataloaders(config)
model = build_model(config).to(DEVICE)
total_params, trainable_params = count_model_parameters(model)

if RUN_TRAINING:
    history = fit(model, train_loader, val_loader, config=config, paths=paths, device=DEVICE)
    checkpoint_path = paths.checkpoints_dir / "best_model.pt"
elif LOAD_EXISTING_BEST_MODEL:
    checkpoint_path = HIGH_SCORE_CHECKPOINT
    history = {}
else:
    checkpoint_path = paths.checkpoints_dir / "best_model.pt"
    history = {}

{
    "checkpoint_path": str(checkpoint_path),
    "output_dir": str(paths.outputs_dir),
    "total_params": total_params,
    "trainable_params": trainable_params,
}

## 结果输出

评估结果会写入 `outputs_runs/project_notebook/evaluation/`，预测结果会写入 `outputs_runs/project_notebook/predictions/`，不会影响 `outputs_submission/`。

In [ ]:
if RUN_EVALUATION:
    eval_model, checkpoint_payload = load_model_from_checkpoint(checkpoint_path, config, DEVICE)
    images, y_true, y_pred = collect_predictions(eval_model, val_loader, device=DEVICE)
    evaluation_summary = save_evaluation_bundle(
        images=images,
        y_true=y_true,
        y_pred=y_pred,
        output_dir=paths.evaluation_dir,
        num_classes=config.num_classes,
    )
else:
    evaluation_summary = None

evaluation_summary

In [ ]:
if RUN_PREDICTION:
    if not EXAM_IMAGE_DIR.exists():
        raise FileNotFoundError(f"考试图片目录不存在: {EXAM_IMAGE_DIR}")
    pred_model, _ = load_model_from_checkpoint(checkpoint_path, config, DEVICE)
    prediction_dataset = PredictionImageDataset(
        EXAM_IMAGE_DIR,
        image_size=config.image_size,
        auto_invert=config.auto_invert,
    )
    prediction_loader = torch.utils.data.DataLoader(
        prediction_dataset,
        batch_size=config.batch_size,
        shuffle=False,
    )
    prediction_rows = []
    with torch.no_grad():
        for batch_images, filenames in prediction_loader:
            logits = predict_with_tta(pred_model, batch_images, config, DEVICE)
            predictions = logits.argmax(dim=1).cpu().tolist()
            prediction_rows.extend(zip(filenames, predictions))
    prediction_csv = paths.predictions_dir / "predictions.csv"
    write_predictions_csv(prediction_rows, prediction_csv)
else:
    prediction_csv = None

prediction_csv